<a href="https://colab.research.google.com/github/10decn/lab-4/blob/main/wnba2021.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
df = pd.read_excel('wnba-shots-2021.xlsx')

Saving wnba-shots-2021.xlsx to wnba-shots-2021.xlsx


In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

df = pd.read_excel('wnba-shots-2021.xlsx')

df.head()

,game_id,game_play_number,desc,shot_type,made_shot,shot_value,coordinate_x,coordinate_y,shooting_team,home_team_name,away_team_name,home_score,away_score,qtr,quarter_seconds_remaining,game_seconds_remaining
0,401391650,4,Shatori Walker-Kimbrough blocks Destanni Hende...,Jump Shot,False,0,37,9,Indiana,Washington,Indiana,0,0,1,571,2371
1,401391650,7,Elena Delle Donne misses two point shot,Turnaround Bank Jump Shot,False,0,12,0,Washington,Washington,Indiana,0,0,1,551,2351
2,401391650,9,Tiffany Mitchell makes 4-foot layup (Destanni ...,Cutting Layup Shot,True,2,29,2,Indiana,Washington,Indiana,0,2,1,538,2338
3,401391650,10,Natasha Cloud makes driving layup,Driving Layup Shot,True,2,22,0,Washington,Washington,Indiana,2,2,1,524,2324
4,401391650,11,Tiffany Mitchell makes 26-foot three point jum...,Jump Shot,True,3,9,21,Indiana,Washington,Indiana,2,5,1,512,2312


In [11]:
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({'Missing Values': missing_values, 'Percent': missing_percent})
print("Missing Values:\n", missing_df[missing_df['Missing Values'] > 0])

duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")

print("\nData types:\n", df.dtypes)

invalid_coords = df[(df['coordinate_x'] < -1000) | (df['coordinate_y'] < -1000)]
print(f"\nInvalid coordinates (x or y < -1000): {len(invalid_coords)}")
invalid_coords[['coordinate_x', 'coordinate_y', 'desc']].head()

Missing Values:
 Empty DataFrame
Columns: [Missing Values, Percent]
Index: []

Number of duplicate rows: 0

Data types:
 game_id                       int64
game_play_number              int64
desc                         object
shot_type                    object
made_shot                      bool
shot_value                    int64
coordinate_x                  int64
coordinate_y                  int64
shooting_team                object
home_team_name               object
away_team_name               object
home_score                    int64
away_score                    int64
qtr                           int64
quarter_seconds_remaining     int64
game_seconds_remaining        int64
dtype: object

Invalid coordinates (x or y < -1000): 8633


,coordinate_x,coordinate_y,desc
7,-214748340,-214748365,Kelsey Mitchell makes free throw 1 of 1
27,-214748340,-214748365,Shakira Austin makes free throw 1 of 1
30,-214748340,-214748365,Shatori Walker-Kimbrough makes free throw 1 of 2
31,-214748340,-214748365,Shatori Walker-Kimbrough makes free throw 2 of 2
36,-214748340,-214748365,Natasha Cloud makes free throw 1 of 2


In [15]:

numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Filled missing in {col} with median: {median_val}")

text_cols = df.select_dtypes(include=['object']).columns
for col in text_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"Filled missing in {col} with mode: {mode_val}")

In [16]:
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

numeric_features = ['coordinate_x', 'coordinate_y', 'quarter_seconds_remaining', 'game_seconds_remaining', 'shot_value']

for col in numeric_features:
    outliers, low, high = detect_outliers_iqr(df, col)
    print(f"\n{col} - Outliers: {len(outliers)}")
    print(f"Bounds: ({low}, {high})")

    df[col] = np.where(df[col] < low, low, df[col])
    df[col] = np.where(df[col] > high, high, df[col])
    print(f"After capping: min={df[col].min()}, max={df[col].max()}")


coordinate_x - Outliers: 8633
Bounds: (-28.0, 60.0)
After capping: min=-28.0, max=50.0

coordinate_y - Outliers: 8730
Bounds: (-21.0, 35.0)
After capping: min=-21.0, max=35.0

quarter_seconds_remaining - Outliers: 0
Bounds: (-329.5, 890.5)
After capping: min=0.0, max=600.0

game_seconds_remaining - Outliers: 0
Bounds: (-1255.0, 3617.0)
After capping: min=0.0, max=2395.0

shot_value - Outliers: 0
Bounds: (-3.0, 5.0)
After capping: min=0.0, max=3.0


In [17]:
features_to_normalize = ['coordinate_x', 'coordinate_y', 'quarter_seconds_remaining', 'game_seconds_remaining']

# Min-Max Scaling
scaler_minmax = MinMaxScaler()
df_minmax = df.copy()
df_minmax[features_to_normalize] = scaler_minmax.fit_transform(df[features_to_normalize])

# Z-score Standardization
scaler_std = StandardScaler()
df_std = df.copy()
df_std[features_to_normalize] = scaler_std.fit_transform(df[features_to_normalize])

print("Min-Max scaled data sample:\n", df_minmax[features_to_normalize].head())
print("\nZ-score standardized data sample:\n", df_std[features_to_normalize].head())

Min-Max scaled data sample:
    coordinate_x  coordinate_y  quarter_seconds_remaining  \
0      0.833333      0.535714                   0.951667   
1      0.512821      0.375000                   0.918333   
2      0.730769      0.410714                   0.896667   
3      0.641026      0.375000                   0.873333   
4      0.474359      0.750000                   0.853333   

   game_seconds_remaining  
0                0.989979  
1                0.981628  
2                0.976200  
3                0.970355  
4                0.965344  

Z-score standardized data sample:
    coordinate_x  coordinate_y  quarter_seconds_remaining  \
0      0.989843      0.410142                   1.661705   
1     -0.077308     -0.209586                   1.546712   
2      0.648354     -0.071869                   1.471966   
3      0.349552     -0.209586                   1.391471   
4     -0.205367      1.236446                   1.322475   

   game_seconds_remaining  
0                

In [18]:

corr_matrix = df[features_to_normalize].corr()
print("Correlation Matrix:\n", corr_matrix)

from sklearn.preprocessing import StandardScaler
scaler_pca = StandardScaler()
scaled_data = scaler_pca.fit_transform(df[features_to_normalize])

pca = PCA()
pca_result = pca.fit_transform(scaled_data)

explained_variance = pca.explained_variance_ratio_
print("\nExplained variance ratio per component:", explained_variance)

cumsum = np.cumsum(explained_variance)
n_components = np.argmax(cumsum >= 0.9) + 1
print(f"Number of components to explain 90% variance: {n_components}")

pca_final = PCA(n_components=n_components)
pca_final_result = pca_final.fit_transform(scaled_data)

print(f"Final PCA shape: {pca_final_result.shape}")

Correlation Matrix:
                            coordinate_x  coordinate_y  \
coordinate_x                   1.000000      0.776566   
coordinate_y                   0.776566      1.000000   
quarter_seconds_remaining      0.102923      0.078248   
game_seconds_remaining         0.093070      0.075359   

                           quarter_seconds_remaining  game_seconds_remaining  
coordinate_x                                0.102923                0.093070  
coordinate_y                                0.078248                0.075359  
quarter_seconds_remaining                   1.000000                0.275831  
game_seconds_remaining                      0.275831                1.000000  

Explained variance ratio per component: [0.45790191 0.30531606 0.18103827 0.05574376]
Number of components to explain 90% variance: 3
Final PCA shape: (41497, 3)
